In [1]:
!pip install transformers torch

In [2]:
from transformers import pipeline

# Model sentimen Bahasa Indonesia siap pakai
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

# Tes pada 1 review
hasil = sentiment_analyzer("Produknya bagus sekali, sangat memuaskan!")
print(hasil)
# Output: [{'label': 'positive', 'score': 0.99...}]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'positive', 'score': 0.9942994117736816}]


In [5]:
import pandas as pd

hasil_df = pd.DataFrame(hasil)
print(hasil_df.columns)
display(hasil_df.head())

Index(['label', 'score'], dtype='object')


,label,score
0,positive,0.994299


In [6]:
import pandas as pd

df = pd.read_csv('/content/data_labeled.csv')
sample = df.head(10).copy()

# Prediksi sentimen
hasil = sentiment_analyzer(sample['teks'].tolist())
sample['prediksi_bert'] = [h['label'] for h in hasil]
sample['confidence']    = [round(h['score'], 3) for h in hasil]

print(sample[['teks', 'sentimen', 'prediksi_bert', 'confidence']])

                                                teks sentimen prediksi_bert  \
0                    gabisa login diwajibkan premium  negatif       neutral   
1                                              jelek  negatif      negative   
2                            ngapa berbayar si? aneh  negatif      negative   
3  tidak suka versi yang sekarang. untuk membuat ...  negatif      negative   
4  semua bayar mana banyak errornya lgi pantesan ...  negatif      negative   
5                                            no free  negatif       neutral   
6                               berbayar untuk login  negatif       neutral   
7                                              bagus  positif      positive   
8       mendaftar pon udah kenak biaya. mengecewakan  negatif      negative   
9         sangat di sayangkan skrng mlah suruh bayar  negatif      negative   

   confidence  
0       0.941  
1       0.999  
2       0.996  
3       0.997  
4       0.999  
5       0.747  
6       0.981  
7 

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Gunakan split yang sama seperti Tugas 3
X = df['teks']  # gunakan teks ASLI (DistilBERT punya tokenizer sendiri)
y = df['sentimen']
_, X_test_raw, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Prediksi pada test set
hasil_bert = sentiment_analyzer(X_test_raw.tolist())
pred_bert  = [h['label'] for h in hasil_bert]

# Sesuaikan label (model output mungkin 'positive'/'negative', dataset mungkin 'positif'/'negatif')
mapping = {'positive': 'positif', 'negative': 'negatif', 'neutral': 'netral'}
pred_bert_map = [mapping.get(p.lower(), p) for p in pred_bert]

print('=== DistilBERT/RoBERTa Indonesia ===')
print(f'Akurasi: {accuracy_score(y_test, pred_bert_map):.4f}')
print('\n', classification_report(y_test, pred_bert_map))

=== DistilBERT/RoBERTa Indonesia ===
Akurasi: 0.7136

               precision    recall  f1-score   support

     negatif       0.70      0.97      0.81       125
      netral       0.00      0.00      0.00         0
     positif       1.00      0.35      0.52        88

    accuracy                           0.71       213
   macro avg       0.57      0.44      0.44       213
weighted avg       0.82      0.71      0.69       213



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
